# Week 22: Multimodal Foundation Models & Vision-Language Architectures
## Notebook 04: Discrete Speech Tokens & Audio Language Modeling from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In modern Voice AI (OpenAI GPT-4o Voice Mode, Meta AudioCraft, ElevenLabs):
- Early voice systems used a disjointed pipeline: Speech-to-Text $\to$ LLM $\to$ Text-to-Speech (loss of emotion, high latency).
- Modern systems treat audio directly as a **language**: converting continuous audio into **Discrete Audio Tokens** using Vector Quantization (VQ-VAE / EnCodec / SoundStream).
- An autoregressive Transformer generates both text tokens and speech tokens simultaneously!

---

### 2. Mathematical Foundations: Vector Quantization (VQ)
Given continuous audio encoder output $z \in \mathbb{R}^D$ and codebook $E = \{e_1, e_2, \dots, e_K\} \subset \mathbb{R}^D$:

1. **Codebook Lookup:**
   $$k^* = \arg\min_k \|z - e_k\|_2$$
   The continuous vector $z$ is replaced by the integer token $k^* \in \{0, 1, \dots, K-1\}$!

2. **Residual Vector Quantization (RVQ):**
   A single codebook causes quantization error. RVQ cascades $M$ codebooks:
   - Stage 1 quantizes $z \to e_{k_1}^{(1)}$, leaves residual $r_1 = z - e_{k_1}^{(1)}$.
   - Stage 2 quantizes $r_1 \to e_{k_2}^{(2)}$, leaves residual $r_2 = r_1 - e_{k_2}^{(2)}$.
   - Audio is represented as an array of $M$ discrete tokens per time frame!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate Continuous Audio Latent Representation
dim = 4
num_frames = 3
z_audio = np.random.randn(num_frames, dim)

# Step 2: Implement Residual Vector Quantization (RVQ) with 2 Stages
class RVQStage:
    def __init__(self, codebook_size=8, dim=4):
        self.codebook = np.random.randn(codebook_size, dim)

    def quantize(self, vectors):
        tokens = []
        quantized = []
        for v in vectors:
            # Nearest neighbor codebook vector
            dists = np.linalg.norm(self.codebook - v, axis=1)
            best_idx = np.argmin(dists)
            tokens.append(best_idx)
            quantized.append(self.codebook[best_idx])
        return np.array(tokens), np.array(quantized)

stage1 = RVQStage(codebook_size=8, dim=dim)
stage2 = RVQStage(codebook_size=8, dim=dim)

# Stage 1: Quantize original audio vectors
tokens_s1, q1 = stage1.quantize(z_audio)
residual_1 = z_audio - q1

# Stage 2: Quantize residual error
tokens_s2, q2 = stage2.quantize(residual_1)
z_reconstructed = q1 + q2

print("--- Discrete Audio Tokenization (EnCodec/SoundStream) ---")
print("Continuous Audio Latents (Frame 0):", np.round(z_audio[0], 3))
print(f"Stage 1 Discrete Token: Code #{tokens_s1[0]} (Reconstructed: {np.round(q1[0], 3)})")
print(f"Stage 2 Residual Token: Code #{tokens_s2[0]} (Reconstructed error: {np.round(q2[0], 3)})")

error_single_stage = np.mean((z_audio - q1) ** 2)
error_rvq_two_stage = np.mean((z_audio - z_reconstructed) ** 2)
print(f"\nSingle Stage Quantization Error: {error_single_stage:.4f}")
print(f"Two-Stage RVQ Quantization Error:    {error_rvq_two_stage:.4f}")
print(f"Error Reduction: {((error_single_stage - error_rvq_two_stage) / error_single_stage) * 100:.1f}%!")

### 3. Step-by-Step Code Walkthrough
1. **Vector Quantization:** Replaces continuous vectors with integer IDs from a learned codebook.
2. **Residual Cascading:** Stage 2 quantizes the leftover error, cutting quantization distortion significantly.
3. **Speech-As-Tokens:** Audio is now represented as discrete tokens (e.g. `[3, 7]`), allowing standard LLM architectures to generate speech natively.

---

### 4. Industry Applications & Trade-offs
- **Real-Time Voice AI (GPT-4o):** Native end-to-end audio-to-audio modeling without intermediate text conversions.
- **AudioCraft (Meta):** Generating music from text prompts using hierarchical discrete audio tokens.

---

### 5. Top Interview Questions & Answers
**Q1: What is the straight-through estimator (STE) in Vector Quantization?**
> **Answer:** The $\arg\min$ codebook selection operation is non-differentiable (zero gradient everywhere). The Straight-Through Estimator (STE) copies gradients directly from the quantized output vector $e_{k^*}$ backward to the continuous input $z$ unchanged ($
abla_z \mathcal{L} \approx \nabla_{e_{k^*}} \mathcal{L}$), allowing backpropagation to train the encoder.

**Q2: What is the difference between Semantic Audio Tokens and Acoustic Audio Tokens?**
> **Answer:** Semantic tokens (e.g. HuBERT, w2v-BERT) capture linguistic phoneme meaning and discard background noise/speaker timbre. Acoustic tokens (e.g. EnCodec, SoundStream) capture full waveform fidelity, prosody, emotion, and speaker identity. Modern voice LLMs combine both.